# CSV statistics

Profiles of each file in the synthetic microbiology dataset. Every count below is computed from the CSVs in this folder. The records are synthetic.

A culture has zero to two isolates, and each isolate has susceptibility results. Keys are `patient_id`, `culture_id`, and `isolate_id`.


## Setup

Loads the six CSV files with pandas. From this folder:

```bash
uv run --with pandas jupyter lab csv_statistics.ipynb
```


In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd()
if not (DATA_DIR / "patients.csv").exists():
    raise FileNotFoundError("Run this notebook from the dataset directory.")

FILES = [
    "patients.csv",
    "wards.csv",
    "antibiotics.csv",
    "cultures.csv",
    "isolates.csv",
    "susceptibility.csv",
]
ID_COLUMNS = {"patient_id", "culture_id", "isolate_id"}
DATE_COLUMNS = {"sample_date", "admission_date"}

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 40)


def load_tables(directory: Path) -> dict[str, pd.DataFrame]:
    """Read every CSV and parse culture dates."""
    frames = {name: pd.read_csv(directory / name) for name in FILES}

    cultures = frames["cultures.csv"]
    cultures["sample_date"] = pd.to_datetime(cultures["sample_date"])
    cultures["admission_date"] = pd.to_datetime(cultures["admission_date"])

    return frames


def missingness(frame: pd.DataFrame) -> pd.DataFrame:
    """Return dtype, missing counts, and distinct-value counts per column."""
    return pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "non_null": frame.notna().sum(),
            "missing": frame.isna().sum(),
            "missing_pct": (frame.isna().mean() * 100).round(2),
            "nunique": frame.nunique(dropna=True),
        }
    )


def numeric_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Describe numeric columns."""
    numeric = frame.select_dtypes(include="number")
    if numeric.empty:
        return numeric

    described = numeric.describe().T
    described["missing"] = numeric.isna().sum()
    return described


def category_counts(series: pd.Series, limit: int = 20) -> pd.DataFrame:
    """Count values, collapsing the tail when there are more than `limit`."""
    counts = series.fillna("(missing)").astype(str).value_counts()
    shown = counts.head(limit)
    table = pd.DataFrame(
        {
            "count": shown.astype(int),
            "pct": (shown / len(series) * 100).round(2),
        }
    )

    if len(counts) > limit:
        other = counts.iloc[limit:].sum()
        table.loc["(other)"] = [int(other), round(other / len(series) * 100, 2)]

    return table


def profile(frame: pd.DataFrame, name: str) -> None:
    """Show size, missingness, numeric summaries, and category counts."""
    print(f"{name}: {len(frame):,} rows, {frame.shape[1]} columns")
    print(f"duplicate rows: {int(frame.duplicated().sum()):,}")
    display(missingness(frame))

    numeric = numeric_summary(frame)
    if not numeric.empty:
        print("numeric summary")
        display(numeric.round(2))

    for column in frame.columns:
        series = frame[column]
        distinct = series.nunique(dropna=True)
        skip_id = column in ID_COLUMNS or column in DATE_COLUMNS
        skip_numeric = pd.api.types.is_numeric_dtype(series) and distinct > 12
        skip_unique = distinct == len(series)
        if skip_id or skip_numeric or skip_unique:
            continue

        print(column)
        display(category_counts(series))


def _check(
    name: str,
    passed: bool,
    detail: str,
) -> dict[str, object]:
    """Build one integrity-check row."""
    return {"check": name, "passed": bool(passed), "detail": detail}


def _missing_count(values: pd.Series, allowed: pd.Series) -> int:
    """Count non-null values that are absent from the allowed set."""
    return int((~values.dropna().isin(set(allowed.dropna()))).sum())


def integrity_report(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Check identifiers, foreign keys, and culture isolate counts."""
    patients = frames["patients.csv"]
    wards = frames["wards.csv"]
    antibiotics = frames["antibiotics.csv"]
    cultures = frames["cultures.csv"]
    isolates = frames["isolates.csv"]
    susceptibility = frames["susceptibility.csv"]

    counted = isolates.groupby("culture_id").size()
    compared = cultures[["culture_id", "n_isolates"]].merge(
        counted.rename("counted"),
        left_on="culture_id",
        right_index=True,
        how="left",
    )
    compared["counted"] = compared["counted"].fillna(0).astype(int)
    isolate_mismatches = int((compared["n_isolates"] != compared["counted"]).sum())

    organism_link = susceptibility.merge(
        isolates[["isolate_id", "organism"]],
        on="isolate_id",
        how="left",
        suffixes=("", "_isolate"),
    )
    organism_mismatches = int(
        (organism_link["organism"] != organism_link["organism_isolate"]).sum()
    )

    missing_culture_patients = _missing_count(
        cultures["patient_id"],
        patients["patient_id"],
    )
    missing_culture_wards = _missing_count(
        cultures["ward_code"],
        wards["ward_code"],
    )
    missing_isolate_cultures = _missing_count(
        isolates["culture_id"],
        cultures["culture_id"],
    )
    missing_susceptibility_isolates = _missing_count(
        susceptibility["isolate_id"],
        isolates["isolate_id"],
    )
    missing_susceptibility_drugs = _missing_count(
        susceptibility["antibiotic_code"],
        antibiotics["antibiotic_code"],
    )

    outpatients = cultures["ward_type"].eq("out")
    inpatients = cultures["ward_type"].eq("in")
    admission_ok = bool(
        cultures.loc[outpatients, "admission_date"].isna().all()
        and cultures.loc[inpatients, "admission_date"].notna().all()
    )

    rows = [
        _check(
            "patients.patient_id is unique",
            patients["patient_id"].is_unique,
            f"{patients['patient_id'].nunique():,} ids",
        ),
        _check(
            "wards.ward_code is unique",
            wards["ward_code"].is_unique,
            f"{len(wards):,} codes",
        ),
        _check(
            "antibiotics.antibiotic_code is unique",
            antibiotics["antibiotic_code"].is_unique,
            f"{len(antibiotics):,} codes",
        ),
        _check(
            "cultures.culture_id is unique",
            cultures["culture_id"].is_unique,
            f"{cultures['culture_id'].nunique():,} ids",
        ),
        _check(
            "isolates.isolate_id is unique",
            isolates["isolate_id"].is_unique,
            f"{isolates['isolate_id'].nunique():,} ids",
        ),
        _check(
            "cultures.patient_id exists in patients",
            missing_culture_patients == 0,
            f"{missing_culture_patients:,} missing",
        ),
        _check(
            "cultures.ward_code exists in wards",
            missing_culture_wards == 0,
            f"{missing_culture_wards:,} missing",
        ),
        _check(
            "isolates.culture_id exists in cultures",
            missing_isolate_cultures == 0,
            f"{missing_isolate_cultures:,} missing",
        ),
        _check(
            "cultures.n_isolates matches isolate rows",
            isolate_mismatches == 0,
            f"{isolate_mismatches:,} mismatches",
        ),
        _check(
            "susceptibility.isolate_id exists in isolates",
            missing_susceptibility_isolates == 0,
            f"{missing_susceptibility_isolates:,} missing",
        ),
        _check(
            "susceptibility.antibiotic_code exists in antibiotics",
            missing_susceptibility_drugs == 0,
            f"{missing_susceptibility_drugs:,} missing",
        ),
        _check(
            "susceptibility.organism matches the isolate",
            organism_mismatches == 0,
            f"{organism_mismatches:,} mismatches",
        ),
        _check(
            "admission_date is filled only for inpatients",
            admission_ok,
            "outpatients blank, inpatients filled",
        ),
    ]
    return pd.DataFrame(rows)


def phenotype_by_organism(isolates: pd.DataFrame) -> pd.DataFrame:
    """Count ESBL, MRSA, VRE, and CPE positives by organism."""
    flags = ["esbl", "mrsa", "vre", "cpe"]
    rows = []

    for organism, group in isolates.groupby("organism"):
        row = {"organism": organism, "n": len(group)}
        for flag in flags:
            row[f"{flag}_n"] = int(group[flag].sum())
            row[f"{flag}_pct"] = round(float(group[flag].mean() * 100), 2)
        rows.append(row)

    table = pd.DataFrame(rows)
    return table.sort_values("n", ascending=False).reset_index(drop=True)


def quantity_by_specimen(isolates: pd.DataFrame, cultures: pd.DataFrame) -> pd.DataFrame:
    """Count isolate quantity values within each specimen type."""
    linked = isolates.merge(
        cultures[["culture_id", "specimen"]],
        on="culture_id",
        how="left",
    )

    return pd.crosstab(
        linked["specimen"],
        linked["quantity"].fillna("(missing)"),
        margins=True,
    )


def resistance_by_antibiotic(
    susceptibility: pd.DataFrame,
    antibiotics: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize susceptible and resistant counts for each antibiotic."""
    labelled = susceptibility.assign(resistant=susceptibility["result"].eq("R"))
    counts = labelled.groupby(["antibiotic_code", "antibiotic"], as_index=False).agg(
        n=("result", "size"),
        resistant=("resistant", "sum"),
    )
    counts["resistant"] = counts["resistant"].astype(int)
    counts["susceptible"] = counts["n"] - counts["resistant"]
    counts["resistant_pct"] = (counts["resistant"] / counts["n"] * 100).round(1)

    order = antibiotics["antibiotic_code"].tolist()
    counts["antibiotic_code"] = pd.Categorical(
        counts["antibiotic_code"],
        categories=order,
        ordered=True,
    )
    return counts.sort_values("antibiotic_code").reset_index(drop=True)


def organism_antibiotic_tables(
    susceptibility: pd.DataFrame,
    isolate_ids: pd.Series,
    drug_order: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return tested counts and percent resistant by organism and antibiotic."""
    subset = susceptibility.loc[
        susceptibility["isolate_id"].isin(set(isolate_ids))
    ].copy()
    subset["resistant"] = subset["result"].eq("R")
    grouped = subset.groupby(["organism", "antibiotic_code"], as_index=False).agg(
        n=("resistant", "size"),
        resistant_n=("resistant", "sum"),
    )
    grouped["resistant_pct"] = (grouped["resistant_n"] / grouped["n"] * 100).round(1)

    counts = grouped.pivot(
        index="organism",
        columns="antibiotic_code",
        values="n",
    ).reindex(columns=drug_order)
    percent = grouped.pivot(
        index="organism",
        columns="antibiotic_code",
        values="resistant_pct",
    ).reindex(columns=drug_order)
    return counts, percent


def first_isolates(isolates: pd.DataFrame, cultures: pd.DataFrame) -> pd.DataFrame:
    """Keep the earliest isolate per patient, organism, and specimen."""
    linked = isolates.merge(
        cultures[["culture_id", "patient_id", "sample_date", "specimen"]],
        on="culture_id",
        how="left",
    )

    linked = linked.sort_values(
        ["patient_id", "organism", "specimen", "sample_date", "isolate_id"],
    )
    return linked.drop_duplicates(
        ["patient_id", "organism", "specimen"],
        keep="first",
    )


def urine_ecoli_resistance_by_sex(
    susceptibility: pd.DataFrame,
    earliest: pd.DataFrame,
    patients: pd.DataFrame,
    drug_order: list[str],
) -> pd.DataFrame:
    """Percent resistant for first urinary E. coli isolates, split by sex."""
    selected = earliest.loc[
        earliest["organism"].eq("Escherichia coli") & earliest["specimen"].eq("urine"),
        ["isolate_id", "patient_id"],
    ]
    selected = selected.merge(
        patients[["patient_id", "sex"]],
        on="patient_id",
        how="left",
    )

    rows = susceptibility.merge(
        selected,
        on="isolate_id",
        how="inner",
    )
    rows = rows.assign(resistant=rows["result"].eq("R"))
    counts = rows.groupby(["antibiotic_code", "sex"], as_index=False).agg(
        n=("resistant", "size"),
        resistant_n=("resistant", "sum"),
    )
    counts["resistant_pct"] = (counts["resistant_n"] / counts["n"] * 100).round(1)

    percent = counts.pivot(
        index="antibiotic_code",
        columns="sex",
        values="resistant_pct",
    ).reindex(drug_order)
    tested = counts.pivot(
        index="antibiotic_code",
        columns="sex",
        values="n",
    ).reindex(drug_order)
    combined = percent.add_suffix("_pct").join(tested.add_suffix("_n"))
    column_order = [
        column
        for column in ["F_pct", "M_pct", "F_n", "M_n"]
        if column in combined.columns
    ]
    return combined.loc[:, column_order].dropna(how="all")


## Overview

Row counts, missing cells, duplicate rows, and identifier checks across the six files.


In [2]:
frames = load_tables(DATA_DIR)

overview = pd.DataFrame(
    [
        {
            "file": name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
            "memory_mb": round(frame.memory_usage(deep=True).sum() / 1_000_000, 2),
        }
        for name, frame in frames.items()
    ]
)
display(overview)
display(integrity_report(frames))


,file,rows,columns,missing_cells,duplicate_rows,memory_mb
0,patients.csv,10000,4,0,0,1.68
1,wards.csv,9,5,0,0,0.00
2,antibiotics.csv,13,5,0,0,0.00
3,cultures.csv,46792,16,27118,0,29.71
4,isolates.csv,24571,9,902,0,6.79
5,susceptibility.csv,172381,8,0,0,81.02


,check,passed,detail
0,patients.patient_id is unique,True,"10,000 ids"
1,wards.ward_code is unique,True,9 codes
2,antibiotics.antibiotic_code is unique,True,13 codes
3,cultures.culture_id is unique,True,"46,792 ids"
4,isolates.isolate_id is unique,True,"24,571 ids"
5,cultures.patient_id exists in patients,True,0 missing
6,cultures.ward_code exists in wards,True,0 missing
7,isolates.culture_id exists in cultures,True,0 missing
8,cultures.n_isolates matches isolate ...,True,0 mismatches
9,susceptibility.isolate_id exists in ...,True,0 missing


## patients.csv

One row per patient: sex, birth year, and home area.


In [3]:
patients = frames["patients.csv"]
profile(patients, "patients.csv")

cultured = frames["cultures.csv"]["patient_id"].nunique()
print(f"patients with at least one culture: {cultured:,} of {len(patients):,}")

birth_decade = (patients["birth_year"] // 10 * 10).astype(str) + "s"
print("birth decade")
display(category_counts(birth_decade).sort_index())


patients.csv: 10,000 rows, 4 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
patient_id,str,10000,0,0.0,10000
sex,str,10000,0,0.0,2
birth_year,int64,10000,0,0.0,100
home_area,str,10000,0,0.0,5


numeric summary


,count,mean,std,min,25%,50%,75%,max,missing
birth_year,10000.0,1981.9,25.36,1927.0,1962.0,1982.0,2003.0,2026.0,0


sex


,count,pct
sex,,
M,5011,50.11
F,4989,49.89


home_area


,count,pct
home_area,,
East,2065,20.65
North,2053,20.53
West,1971,19.71
South,1959,19.59
Centre,1952,19.52


patients with at least one culture: 8,829 of 10,000
birth decade


,count,pct
birth_year,,
1920s,124,1.24
1930s,380,3.80
1940s,603,6.03
1950s,1107,11.07
1960s,1195,11.95
1970s,1262,12.62
1980s,1241,12.41
1990s,1161,11.61
2000s,1194,11.94


## wards.csv

Ward code list. `ward_type` is `in` or `out`. `institution` is `hospital` or `primary_care`.


In [4]:
wards = frames["wards.csv"]
profile(wards, "wards.csv")
display(wards)


wards.csv: 9 rows, 5 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
ward_code,str,9,0,0.0,9
ward_fi,str,9,0,0.0,9
ward_en,str,9,0,0.0,9
ward_type,str,9,0,0.0,2
institution,str,9,0,0.0,2


ward_type


,count,pct
ward_type,,
in,6,66.67
out,3,33.33


institution


,count,pct
institution,,
hospital,7,77.78
primary_care,2,22.22


,ward_code,ward_fi,ward_en,ward_type,institution
0,ED,Päivystys,Emergency department,out,hospital
1,MED-A,Sisätaudit A,Internal medicine A,in,hospital
2,MED-B,Sisätaudit B,Internal medicine B,in,hospital
3,SURG,Kirurgia,Surgery,in,hospital
4,URO,Urologia,Urology,in,hospital
5,ICU,Teho-osasto,Intensive care unit,in,hospital
6,GER,Geriatria,Geriatrics,in,hospital
7,PHC-N,Terveysasema Pohjoinen,Health centre North,out,primary_care
8,PHC-S,Terveysasema Etelä,Health centre South,out,primary_care


## antibiotics.csv

Antibiotic code list. `reported_for` is `all` or `urine`. Urinary agents are reported for urine cultures.


In [5]:
antibiotics = frames["antibiotics.csv"]
profile(antibiotics, "antibiotics.csv")
display(antibiotics)


antibiotics.csv: 13 rows, 5 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
antibiotic,str,13,0,0.0,13
antibiotic_code,str,13,0,0.0,13
antibiotic_fi,str,13,0,0.0,13
class,str,13,0,0.0,11
reported_for,str,13,0,0.0,2


class


,count,pct
class,,
penicillins,2,15.38
cephalosporins 3rd gen,2,15.38
penicillins + BLI,1,7.69
MRSA screen,1,7.69
cephalosporins 2nd gen,1,7.69
carbapenems,1,7.69
fluoroquinolones,1,7.69
trimethoprim,1,7.69
nitrofurans,1,7.69


reported_for


,count,pct
reported_for,,
all,10,76.92
urine,3,23.08


,antibiotic,antibiotic_code,antibiotic_fi,class,reported_for
0,ampicillin,AMP,ampisilliini,penicillins,all
1,mecillinam,MEC,mesillinaami,penicillins,urine
2,piperacillin-tazobactam,TZP,piperasilliini-tatsobaktaami,penicillins + BLI,all
3,cefoxitin (screen),FOX,kefoksitiini (seulonta),MRSA screen,all
4,cefuroxime,CXM,kefuroksiimi,cephalosporins 2nd gen,all
5,ceftriaxone,CRO,keftriaksoni,cephalosporins 3rd gen,all
6,ceftazidime,CAZ,keftatsidiimi,cephalosporins 3rd gen,all
7,meropenem,MEM,meropeneemi,carbapenems,all
8,ciprofloxacin,CIP,siprofloksasiini,fluoroquinolones,all
9,trimethoprim,TMP,trimetopriimi,trimethoprim,urine


## cultures.csv

One row per culture. `result` is `positive`, `negative`, or `mixed_growth`. Inpatients have an admission date. Outpatient admission dates are blank.

Hospital day 1 is the admission date. A sample on hospital day 3 or later is the usual proxy for a hospital-onset finding.


In [6]:
cultures = frames["cultures.csv"]
profile(cultures, "cultures.csv")

print(
    f"sample dates: {cultures['sample_date'].min().date()} "
    f"to {cultures['sample_date'].max().date()}"
)

print("cultures per ISO week")
weekly = cultures.groupby(["iso_year", "iso_week"]).size()
display(weekly.describe().round(1).to_frame("cultures"))

print("cultures per patient, among patients who have a culture")
per_patient = cultures.groupby("patient_id").size()
display(per_patient.describe().round(2).to_frame("cultures"))

age_bands = pd.cut(
    cultures["age_years"],
    bins=[0, 18, 65, 80, 100],
    labels=["0–17", "18–64", "65–79", "80–99"],
    right=False,
)
print("age band")
band_counts = age_bands.value_counts(sort=False).rename("count").to_frame()
band_counts["pct"] = (band_counts["count"] / band_counts["count"].sum() * 100).round(1)
display(band_counts)

print("age by specimen")
display(cultures.groupby("specimen")["age_years"].describe().round(1))

print("n_isolates by result")
display(
    pd.crosstab(
        cultures["result"],
        cultures["n_isolates"],
        margins=True,
    )
)

print("result by specimen (counts)")
display(
    pd.crosstab(
        cultures["specimen"],
        cultures["result"],
        margins=True,
    )
)
print("result by specimen (row %)")
display(
    pd.crosstab(
        cultures["specimen"],
        cultures["result"],
        normalize="index",
    ).mul(100).round(1)
)

print("result by ward type (row %)")
display(
    pd.crosstab(
        cultures["ward_type"],
        cultures["result"],
        normalize="index",
    ).mul(100).round(1)
)

print("specimen by ward")
display(
    pd.crosstab(
        cultures["ward_code"],
        cultures["specimen"],
        margins=True,
    )
)

inpatients = cultures.loc[cultures["ward_type"].eq("in")].copy()
inpatients["hospital_day"] = (
    (inpatients["sample_date"] - inpatients["admission_date"]).dt.days + 1
)
onset = inpatients["hospital_day"].ge(3)
print(
    f"inpatient cultures: {len(inpatients):,}; "
    f"hospital day 3 or later: {int(onset.sum()):,} ({onset.mean() * 100:.1f}%)"
)
print(
    "admission dates: "
    f"{inpatients['admission_date'].min().date()} "
    f"to {inpatients['admission_date'].max().date()}"
)
print("hospital day (1 = admission date)")
display(inpatients["hospital_day"].describe().round(1).to_frame("hospital_day"))


cultures.csv: 46,792 rows, 16 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
culture_id,str,46792,0,0.00,46792
patient_id,str,46792,0,0.00,8829
age_years,int64,46792,0,0.00,100
sample_date,datetime64[us],46792,0,0.00,1092
iso_year,int64,46792,0,0.00,4
iso_week,int64,46792,0,0.00,52
ward_code,str,46792,0,0.00,9
ward_fi,str,46792,0,0.00,9
ward_en,str,46792,0,0.00,9
ward_type,str,46792,0,0.00,2


numeric summary


,count,mean,std,min,25%,50%,75%,max,missing
age_years,46792.0,59.50,24.66,0.0,40.0,66.0,77.0,99.0,0
iso_year,46792.0,2024.76,0.93,2023.0,2024.0,2025.0,2026.0,2026.0,0
iso_week,46792.0,26.83,14.64,1.0,15.0,27.0,39.0,52.0,0
n_isolates,46792.0,0.53,0.59,0.0,0.0,0.0,1.0,2.0,0


iso_year


...

result,mixed_growth,negative,positive
ward_type,,,
in,6.5,48.6,44.9
out,8.7,42.0,49.3


specimen by ward


specimen,blood,urine,wound,All
ward_code,,,,
ED,2429,5986,1420,9835
GER,361,2296,568,3225
ICU,1092,312,545,1949
MED-A,1259,2668,330,4257
MED-B,1007,1931,0,2938
PHC-N,0,7780,833,8613
PHC-S,0,7790,880,8670
SURG,489,646,2494,3629
URO,337,3339,0,3676


inpatient cultures: 19,674; hospital day 3 or later: 13,960 (71.0%)
admission dates: 2023-09-11 to 2026-09-26
hospital day (1 = admission date)


,hospital_day
count,19674.0
mean,6.9
std,6.2
min,1.0
25%,2.0
50%,4.0
75%,10.0
max,22.0


## isolates.csv

One row per isolate. `quantity` is a urine CFU/ml band or a wound score (`+` to `+++`). Blood isolates have no quantity. Phenotype flags are 0/1.


In [7]:
isolates = frames["isolates.csv"]
profile(isolates, "isolates.csv")

phenotype_columns = ["esbl", "mrsa", "vre", "cpe"]
print("phenotype positives")
display(
    pd.DataFrame(
        {
            "positive": isolates[phenotype_columns].sum().astype(int),
            "pct": (isolates[phenotype_columns].mean() * 100).round(2),
        }
    )
)

print("phenotypes by organism")
display(phenotype_by_organism(isolates))

print("quantity by specimen")
display(quantity_by_specimen(isolates, cultures))


isolates.csv: 24,571 rows, 9 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
isolate_id,str,24571,0,0.00,24571
culture_id,str,24571,0,0.00,22192
isolate_no,int64,24571,0,0.00,2
organism,str,24571,0,0.00,9
quantity,str,23669,902,3.67,6
esbl,int64,24571,0,0.00,2
mrsa,int64,24571,0,0.00,2
vre,int64,24571,0,0.00,2
cpe,int64,24571,0,0.00,2


numeric summary


,count,mean,std,min,25%,50%,75%,max,missing
isolate_no,24571.0,1.10,0.30,1.0,1.0,1.0,1.0,2.0,0
esbl,24571.0,0.02,0.14,0.0,0.0,0.0,0.0,1.0,0
mrsa,24571.0,0.00,0.07,0.0,0.0,0.0,0.0,1.0,0
vre,24571.0,0.00,0.01,0.0,0.0,0.0,0.0,1.0,0
cpe,24571.0,0.00,0.02,0.0,0.0,0.0,0.0,1.0,0


isolate_no


...

,positive,pct
esbl,517,2.10
mrsa,111,0.45
vre,5,0.02
cpe,12,0.05


phenotypes by organism


,organism,n,esbl_n,esbl_pct,mrsa_n,mrsa_pct,vre_n,vre_pct,cpe_n,cpe_pct
0,Escherichia coli,11534,396,3.43,0,0.00,0,0.00,2,0.02
1,Staphylococcus aureus,3017,0,0.00,111,3.68,0,0.00,0,0.00
2,Enterococcus faecalis,2385,0,0.00,0,0.00,5,0.21,0,0.00
3,Klebsiella pneumoniae,2384,121,5.08,0,0.00,0,0.00,4,0.17
4,Proteus mirabilis,1725,0,0.00,0,0.00,0,0.00,2,0.12
5,Pseudomonas aeruginosa,1653,0,0.00,0,0.00,0,0.00,0,0.00
6,Citrobacter freundii,874,0,0.00,0,0.00,0,0.00,3,0.34
7,Klebsiella oxytoca,621,0,0.00,0,0.00,0,0.00,1,0.16
8,Staphylococcus saprophyticus,378,0,0.00,0,0.00,0,0.00,0,0.00


quantity by specimen


quantity,(missing),+,++,+++,10^3,10^4,>=10^5,All
specimen,,,,,,,,
blood,902,0,0,0,0,0,0,902
urine,0,0,0,0,1875,3759,13116,18750
wound,0,1484,1989,1446,0,0,0,4919
All,902,1484,1989,1446,1875,3759,13116,24571


## susceptibility.csv

One row per isolate and antibiotic. `result` is `S` or `R`.

Percent resistant is the share of `R` results. The all-rows tables count every result in this file. The first-isolate tables keep the earliest isolate per patient, organism, and specimen over the whole date range, and leave a percentage blank when fewer than 30 isolates were tested. `<NA>` is an organism–antibiotic pair that is not reported. Urinary *E. coli* is also split by sex.


In [8]:
susceptibility = frames["susceptibility.csv"]
profile(susceptibility, "susceptibility.csv")

resistant_n = int(susceptibility["result"].eq("R").sum())
print(
    f"resistant rows: {resistant_n:,} of {len(susceptibility):,} "
    f"({resistant_n / len(susceptibility) * 100:.1f}%)"
)

print("results per isolate")
per_isolate = susceptibility.groupby("isolate_id").size()
display(per_isolate.describe().round(2).to_frame("results"))

print("resistance by antibiotic (all rows)")
display(resistance_by_antibiotic(susceptibility, antibiotics))

drug_order = antibiotics["antibiotic_code"].tolist()
organism_order = isolates["organism"].value_counts().index.tolist()
counts, percent = organism_antibiotic_tables(
    susceptibility,
    susceptibility["isolate_id"],
    drug_order,
)
counts = counts.reindex(organism_order).astype("Int64")
percent = percent.reindex(organism_order)
print("isolates tested, by organism and antibiotic")
display(counts)
print("percent resistant, by organism and antibiotic (all rows)")
display(percent)

earliest = first_isolates(isolates, cultures)
print(
    f"first isolates (one per patient, organism, and specimen): "
    f"{len(earliest):,} of {len(isolates):,}"
)
first_counts, first_percent = organism_antibiotic_tables(
    susceptibility,
    earliest["isolate_id"],
    drug_order,
)
first_counts = first_counts.reindex(organism_order).astype("Int64")
first_percent = first_percent.reindex(organism_order).where(first_counts.ge(30))
print("first isolates tested, by organism and antibiotic")
display(first_counts)
print("first-isolate percent resistant; blank when fewer than 30 isolates were tested")
display(first_percent)

urine_first = earliest.loc[
    earliest["organism"].eq("Escherichia coli") & earliest["specimen"].eq("urine")
]
print(f"first urinary E. coli isolates: {len(urine_first):,}")
print("first urinary E. coli, percent resistant by sex")
display(
    urine_ecoli_resistance_by_sex(
        susceptibility,
        earliest,
        patients,
        drug_order,
    )
)


susceptibility.csv: 172,381 rows, 8 columns
duplicate rows: 0


,dtype,non_null,missing,missing_pct,nunique
isolate_id,str,172381,0,0.0,24571
culture_id,str,172381,0,0.0,22192
organism,str,172381,0,0.0,9
antibiotic,str,172381,0,0.0,13
antibiotic_code,str,172381,0,0.0,13
antibiotic_fi,str,172381,0,0.0,13
result,str,172381,0,0.0,2
guideline,str,172381,0,0.0,4


organism


,count,pct
organism,,
Escherichia coli,101061,58.63
Klebsiella pneumoniae,20409,11.84
Proteus mirabilis,14124,8.19
Staphylococcus aureus,10309,5.98
Citrobacter freundii,7059,4.09
Pseudomonas aeruginosa,6612,3.84
Enterococcus faecalis,6564,3.81
Klebsiella oxytoca,5487,3.18
Staphylococcus saprophyticus,756,0.44


antibiotic


...

antibiotic_code,AMP,MEC,TZP,FOX,CXM,CRO,CAZ,MEM,CIP,TMP,NIT,VAN,CLI
organism,,,,,,,,,,,,,
Escherichia coli,5948,5072,5948,<NA>,5948,5948,<NA>,5948,5948,5072,5072,<NA>,<NA>
Staphylococcus aureus,<NA>,<NA>,<NA>,2650,<NA>,<NA>,<NA>,<NA>,<NA>,582,582,2650,2650
Enterococcus faecalis,2052,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1486,2052,<NA>
Klebsiella pneumoniae,2012,1668,2012,<NA>,2012,2012,<NA>,2012,2012,1668,1668,<NA>,<NA>
Proteus mirabilis,1548,1100,1548,<NA>,1548,1548,<NA>,1548,1548,1100,1100,<NA>,<NA>
Pseudomonas aeruginosa,<NA>,<NA>,1534,<NA>,<NA>,<NA>,1534,1534,1534,<NA>,<NA>,<NA>,<NA>
Citrobacter freundii,838,573,838,<NA>,838,838,<NA>,838,838,573,573,<NA>,<NA>
Klebsiella oxytoca,576,542,576,<NA>,576,576,<NA>,576,576,542,542,<NA>,<NA>
Staphylococcus saprophyticus,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,359,359,<NA>,<NA>


first-isolate percent resistant; blank when fewer than 30 isolates were tested


antibiotic_code,AMP,MEC,TZP,FOX,CXM,CRO,CAZ,MEM,CIP,TMP,NIT,VAN,CLI
organism,,,,,,,,,,,,,
Escherichia coli,36.4,2.9,4.7,NaN,8.6,4.3,NaN,0.0,8.8,18.3,0.4,NaN,NaN
Staphylococcus aureus,NaN,NaN,NaN,3.6,NaN,NaN,NaN,NaN,NaN,5.2,1.2,0.0,4.6
Enterococcus faecalis,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9,0.2,NaN
Klebsiella pneumoniae,100.0,5.3,7.4,NaN,12.5,7.2,NaN,0.2,9.5,15.3,33.5,NaN,NaN
Proteus mirabilis,16.5,2.2,1.1,NaN,4.3,1.2,NaN,0.1,5.8,19.1,100.0,NaN,NaN
Pseudomonas aeruginosa,NaN,NaN,8.9,NaN,NaN,NaN,7.9,5.5,11.7,NaN,NaN,NaN,NaN
Citrobacter freundii,100.0,11.3,17.1,NaN,40.3,14.3,NaN,0.4,6.0,12.7,4.5,NaN,NaN
Klebsiella oxytoca,100.0,5.2,5.6,NaN,9.5,2.1,NaN,0.2,5.7,7.4,8.5,NaN,NaN
Staphylococcus saprophyticus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.3,1.7,NaN,NaN


first urinary E. coli isolates: 5,072
first urinary E. coli, percent resistant by sex


sex,F_pct,M_pct,F_n,M_n
antibiotic_code,,,,
AMP,34.3,37.7,2956.0,2116.0
MEC,2.7,3.1,2956.0,2116.0
TZP,4.3,4.8,2956.0,2116.0
CXM,6.7,10.8,2956.0,2116.0
CRO,2.9,5.9,2956.0,2116.0
MEM,0.0,0.0,2956.0,2116.0
CIP,6.1,12.6,2956.0,2116.0
TMP,17.6,19.4,2956.0,2116.0
NIT,0.4,0.3,2956.0,2116.0
